In [2]:
%load_ext autoreload
%autoreload 2

from src.features.build_features import load_and_merge

df = load_and_merge()
print("shape:", df.shape)
print("has_identity rate:", round(df["has_identity"].mean(), 4))
print("fraud rate:", round(df["isFraud"].mean(), 4))

shape: (590540, 435)
has_identity rate: 0.2442
fraud rate: 0.035


In [3]:
from src.features.build_features import time_split

train, valid = time_split(df)

print("train:", train.shape, "| fraud rate:", round(train["isFraud"].mean(), 4))
print("valid:", valid.shape, "| fraud rate:", round(valid["isFraud"].mean(), 4))
print("no time overlap:", train["TransactionDT"].max() <= valid["TransactionDT"].min())

train: (472432, 435) | fraud rate: 0.0351
valid: (118108, 435) | fraud rate: 0.0344
no time overlap: True


In [5]:
from src.features.build_features import add_features

df = add_features(df)
print(df["n_missing"].describe().round(1))

count    590540.0
mean        195.6
std          49.0
min          24.0
25%         208.0
50%         211.0
75%         229.0
max         340.0
Name: n_missing, dtype: float64


In [6]:
print(df.groupby("has_identity")["n_missing"].mean().round(1))

has_identity
0    221.1
1    116.7
Name: n_missing, dtype: float64


In [7]:
# re-split, so train/valid include the new column
train, valid = time_split(df)
print("n_missing in train:", "n_missing" in train.columns)

n_missing in train: True


In [10]:
obj_cols = df.select_dtypes("object", "str").columns
print("text columns:", len(obj_cols))
print(df[obj_cols].nunique().sort_values(ascending=False))

text columns: 0
Series([], dtype: float64)


In [11]:
for col in ["ProductCD", "card4", "card6", "P_emaildomain", "DeviceInfo"]:
    print(col, "->", df[col].value_counts(dropna=False).head(5).to_dict())

ProductCD -> {'W': 439670, 'C': 68519, 'R': 37699, 'H': 33024, 'S': 11628}
card4 -> {'visa': 384767, 'mastercard': 189217, 'american express': 8328, 'discover': 6651, nan: 1577}
card6 -> {'debit': 439938, 'credit': 148986, nan: 1571, 'debit or credit': 30, 'charge card': 15}
P_emaildomain -> {'gmail.com': 228355, 'yahoo.com': 100934, nan: 94456, 'hotmail.com': 45250, 'anonymous.com': 36998}
DeviceInfo -> {nan: 471874, 'Windows': 47722, 'iOS Device': 19782, 'MacOS': 12573, 'Trident/7.0': 7440}


In [12]:
from src.features.build_features import encode_categoricals

before = df.memory_usage(deep=True).sum() / 1e6
df = encode_categoricals(df)
after = df.memory_usage(deep=True).sum() / 1e6

print(f"memory: {before:.0f} MB -> {after:.0f} MB")
print("text columns left:", len(df.select_dtypes(include=["object", "str"]).columns))
print("category columns:", len(df.select_dtypes("category").columns))
print(df["card4"].cat.categories.tolist())

memory: 1127 MB -> 971 MB
text columns left: 0
category columns: 31
['american express', 'discover', 'mastercard', 'visa']


In [1]:
%load_ext autoreload
%autoreload 2

from src.features.build_features import build_dataset

X_train, y_train, X_valid, y_valid = build_dataset()

print("X_train:", X_train.shape, "| y_train:", y_train.shape)
print("X_valid:", X_valid.shape, "| y_valid:", y_valid.shape)
print("answer leaked into X?", "isFraud" in X_train.columns)
print("time column in X?", "TransactionDT" in X_train.columns)

X_train: (472432, 433) | y_train: (472432,)
X_valid: (118108, 433) | y_valid: (118108,)
answer leaked into X? False
time column in X? False


In [2]:
import numpy as np
from src.training import evaluate

# Fake model 1: random guesses
rng = np.random.default_rng(42)
print("random:   ", evaluate(y_valid, rng.random(len(y_valid))))

# Fake model 2: "fewer blanks = more suspicious" (from our EDA)
print("n_missing:", evaluate(y_valid, -X_valid["n_missing"]))

random:    {'roc_auc': 0.5, 'auc_pr': 0.0347, 'recall_at_p90': 0.0}
n_missing: {'roc_auc': 0.6997, 'auc_pr': 0.0847, 'recall_at_p90': 0.0}
